## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para disponer de persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Bajar datasets si hace falta

In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

In [3]:
## Para correr con extensión de Colab en Visual Studio Code

In [4]:
import plotly.io as pio

# Set the renderer explicitly for VS Code Notebooks
pio.renderers.default = "vscode"  # alternative: 'notebook_mimetype'


# 1  Modelo AutoARIMA

## 1.1 Init Experimento

In [5]:
# instalacion de paquetes que NO vienen por default en Colab
!pip install uv
!uv pip install -q kaggle
!uv pip install -q pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.1/25.1 MB 61.4 MB/s eta 0:00:00:00:0100:01


In [6]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [7]:
import os as os
import numpy as np
import polars as pl
from pmdarima import auto_arima

import warnings
warnings.filterwarnings("ignore")

Por favor, cargar aqui SU semilla primigenia

In [ ]:
# defino los parametros
PARAM = {'experimento':'AA01',
  'kaggle_competition':'labo-iii-2026-ba',
  'semilla_primigenia':102191
}

In [ ]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/AA01


## 1.2 Init AutoARIMA

In [10]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [11]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [12]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


In [ ]:
display(tb_ventas)

product_id,periodo,tn
i64,i64,f64
20001,201701,934.77222
20001,201702,798.0162
20001,201703,1303.35771
20001,201704,1069.9613
20001,201705,1502.20132
…,…,…
21276,201908,0.01265
21276,201909,0.01856
21276,201910,0.02079


Opcion de empiojar el dataset, agregando ruido relativo a las ventas
<br>Un Experimento no se le niega a nadie

In [ ]:
empiojar=False
empiojar_ruido=0.3

if empiojar:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=empiojar_ruido, size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


In [15]:
display(tb_ventas)

product_id,periodo,tn
i64,i64,f64
20001,201701,934.77222
20001,201702,798.0162
20001,201703,1303.35771
20001,201704,1069.9613
20001,201705,1502.20132
…,…,…
21276,201908,0.01265
21276,201909,0.01856
21276,201910,0.02079


In [16]:
# donde voy a guardar el resultado final
arch_tb_final = 'tb_final.csv'

try:
    # Attempt to load the existing file
    tb_final = pl.read_csv(arch_tb_final)
except FileNotFoundError:
    # donde voy a guardar el resultado final
    tb_final = tb_apredecir.clone()
    tb_final = tb_final.with_columns(pl.lit(None).cast(pl.Float64).alias('tn'))


tb_final = tb_final.sort(["product_id"])


# cuento cuantos registros NO puedieron calcularse
# en mi corrida esto da  209

tb_final["tn"].is_null().sum()


780

In [ ]:
correrdeCero=True

if correrdeCero:
  tb_final = tb_apredecir.clone()
  tb_final = tb_final.with_columns(pl.lit(None).cast(pl.Float64).alias('tn'))


tb_final = tb_final.sort(["product_id"])

In [18]:
display(tb_final)

product_id,tn
i64,f64
20001,null
20002,null
20003,null
20004,null
20005,null
…,…
21263,null
21265,null
21266,null


## 1.3 Primera pasada AutoARIMA

Tengo en cuenta la estacionalidad de 12 periodos
<br> muchos van a quedar SIN  calcular, porque no tienen suficiente historia
<br> Estra primera pasada lleva 55 minutos !

In [19]:
# solo los productos que faltan

productos = tb_final.filter(
    pl.col("tn").is_null()
).select("product_id" ).get_column("product_id").to_list()


# Es fundamental que tb_ventas este ORDENADO
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

for producto in productos:

  print(producto, end=' ')
  tn_values = tb_ventas.filter(pl.col("product_id") == producto).select(["tn"])

  try:
    modelo = auto_arima(
      tn_values,
      seasonal=True,
      m=12, # estacinalidad cada 12
      stepwise=True,
      suppress_warnings=True,
      error_action="ignore",
      max_p=3, max_q=3,
      max_P=2, max_Q=2,
      random=True,
      random_state=PARAM['semilla_primigenia'],
      n_fits=10 # cantidad de iteraciones de busqueda AUTO
    )

    # predigo el periodo+1 y periodo+2
    forecast = modelo.predict(n_periods=2)
    mesmasdos = forecast[1]  # el segundo elemento del vector

    tb_final = tb_final.with_columns(
       pl.when(pl.col("product_id") == producto)
      .then(mesmasdos if mesmasdos >=0  else 0 )
      .otherwise(pl.col("tn")).alias("tn")
    )

  except Exception as e:
    print(f"  ERROR for  {producto} ")

20001 20002 20003 20004 20005 20006 20007 20008 20009 20010 20011 20012 20013 20014 20015 20016 20017 20018 20019 20020 20021 20022 20023 20024 20025 20026 20027 20028 20029 20030 20031 20032   ERROR for  20032 
20033 20035 20037 20038 20039 20041 20042 20043 20044 20045 20046 20047 20049 20050 20051 20052 20053 20054 20055 20056 20057 20058 20059 20061 20062 20063 20065 20066 20067 20068 20069 20070 20071 20072 20073 20074 20075 20076 20077 20079 20080 20081 20082 20084 20085   ERROR for  20085 
20086 20087 20089   ERROR for  20089 
20090 20091 20092 20093 20094 20095 20096 20097 20099 20100 20101 20102 20103 20106 20107 20108 20109 20111 20112 20114 20116 20117 20118 20119 20120 20121 20122 20123 20124 20125 20126 20127   ERROR for  20127 
20129 20130 20132 20133 20134 20135 20137 20138 20139 20140 20142 20143 20144 20145 20146 20148 20150   ERROR for  20150 
20151 20152 20153 20155 20157 20158 20159 20160 20161 20162 20164 20166 20167 20168 20170 20174   ERROR for  20174 
20175 2017

In [20]:
# cuento cuantos registros NO puedieron calcularse
# en mi corrida esto da ~ 210

tb_final["tn"].is_null().sum()

208

In [21]:
arch_tb_final

'tb_final.csv'

In [22]:
# grabo la historia

tb_final.write_csv(arch_tb_final)

## 1.4  Segunda Pasada AutoARIMA

Ahora NO le indico estacionalidad de 12 periodos

Me quedaron prodcutos que NO puede construir el modelo ARIMA debido a no tener suficiente historia para la estacionalidad de 12 meses.
<br> Ahora  quito la condicion de estacionalidad y me encomiendo a la diosa del forecasting

In [23]:
# ahora SIN  estacionalidad

# solo los productos que faltan
productos = tb_final.filter(
    pl.col("tn").is_null()
).select("product_id" ).get_column("product_id").to_list()

# Es fundamental que tb_ventas este ORDENADO
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

for producto in productos:

  print(producto, end=' ')
  tn_values = tb_ventas.filter(pl.col("product_id") == producto).select(["tn"])

  try:
    modelo = auto_arima(
      tn_values,
      seasonal= False,  # Sin estacionalidad
      stepwise=True,
      suppress_warnings=True,
      error_action="ignore",
      max_p=3, max_q=3,
      max_P=2, max_Q=2,
      random=True,
      random_state=PARAM['semilla_primigenia'],
      n_fits=10
    )

    forecast = modelo.predict(n_periods=2)
    mesmasdos = forecast[1]

    tb_final = tb_final.with_columns(
       pl.when(pl.col("product_id") == producto)
      .then(mesmasdos if mesmasdos >=0  else 0)
      .otherwise(pl.col("tn")).alias("tn")
    )

  except Exception as e:
    print(f"  ERROR for  {producto} ")

20032 20085 20089 20127 20150 20174 20202 20203 20210 20213 20218 20236 20257 20261 20286 20297 20306 20323 20344 20355 20368 20378 20387 20408 20414 20440 20442 20458 20459 20460 20476 20477 20481 20491 20495 20496 20503 20510 20521 20522 20523 20525 20526 20527 20531 20537 20540 20547 20548 20552 20553 20558 20559 20569 20574 20575 20577 20580 20589 20592 20593 20603 20611 20615 20620 20621 20623 20627 20633 20638 20649 20657 20659 20662 20673 20674 20679 20681 20682 20686 20689 20691 20694 20700 20703 20709 20711 20719 20720 20732 20741 20746 20754 20757 20758 20762 20772 20774 20777 20783 20785 20795 20811 20815 20817 20822 20827 20832 20835 20845 20853 20859 20886 20899 20902 20904 20907 20908 20910 20912 20917 20920 20924 20927 20928 20932 20933 20942 20946 20953 20962 20966 20967 20968 20975 20987 20990 20995 20997 21001 21006 21007 21022 21033 21035 21037 21039 21042 21044 21056 21058 21064 21073 21074 21079 21084 21086 21087 21092 21093 21097 21099 21105 21109 21110 21111 2111

In [24]:
# cuento cuantos registros NO puedieron calcularse aun en este segundo intento
# a mi me quedaron sin calcular  CERO
tb_final["tn"].is_null().sum()

0

In [25]:
# vuelvo a grabar
tb_final.write_csv(arch_tb_final)

## 1.5 Submit a Kaggle

In [26]:
os.getcwd()

'/content/.drive/MyDrive/labo3/exp/AA01'

In [27]:
# Submit a Kaggle
if not empiojar:
  archivo= "autoARIMA.csv"
  mensaje= "autoARIMA DOS fases"
else:
  archivo= "autoARIMA_empiojado.csv"
  mensaje= "autoARIMA logEMPIOJADO al " + str(empiojar_ruido) + " ,dos fases"

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )